# A04 — cifar10: dữ liệu và so sánh đã lưu

Các framework dùng hai môi trường WSL riêng. Mã huấn luyện ở `training/cifar10_keras.ipynb` và `training/cifar10_torch.ipynb`, cùng `training/cifar10_scratch.ipynb`. Notebook này đọc kết quả gốc; Run All không train/TEST. Toàn bộ notebook huấn luyện nguyên byte và output cũ nằm trong provenance.


In [ ]:
# Migration r2: select this assignment's original logical paths, preserving frozen source.
from pathlib import Path as _Path
import sys as _sys
_repo = next(p for p in (_Path.cwd(), *_Path.cwd().parents) if (p / 'ASSIGN_REORGANIZATION_EXECUTION_PLAN.md').exists())
_sys.path.insert(0, str(_repo / '_shared'))
from assignment_runtime import activate as _activate
_migration = _activate('A04', 'torch')
_migration


# 1. Assignment Overview


Notebook này xây pipeline Assignment 04 cho CIFAR-10 image classification. CIFAR-10 là dataset ảnh RGB 32x32 với 10 classes, khó hơn MNIST vì object có màu, background phức tạp và variation lớn hơn.

**Nhận xét kết quả.**

- Notebook dung CIFAR-10 tu archive chinh thuc cua University of Toronto, voi 50,000 anh train goc va 10,000 anh test.
- Representation la anh RGB kich thuoc 32 x 32 x 3, nen CNN la lua chon phu hop vi co cau truc khong gian va kenh mau that.
- Day la bai toan classification 10 lop: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.

# 2. Environment Setup


Import thư viện, cố định seed, kiểm tra version TensorFlow/PyTorch và xác định CPU/GPU device. Output không in path cá nhân.

**Nhận xét kết quả.**

- Device output cho biết notebook chạy trên CPU hay GPU.
- Seed được cố định để split, initialization và DataLoader reproducible hơn giữa các lần chạy.

# 3. Load Dataset


Load CIFAR-10 từ archive local trong `datasets/cifar10/`. Nếu archive chưa có, code có cơ chế tải lại từ source gốc University of Toronto.

**Nhận xét kết quả.**

- Dataset duoc doc truc tiep tu `datasets/cifar10/cifar-10-python.tar.gz`, khong extract them ban duplicate.
- Shape train/test xac nhan CIFAR-10 chuan: 50,000 train, 10,000 test, moi anh co kich thuoc 32 x 32 RGB.
- Label duoc doc tu metadata trong archive thay vi tu doan thu cong trong notebook.

# 4. Dataset Inspection


Kiểm tra class distribution, RGB channels, sample grid và một ảnh cụ thể trước preprocessing.

**Nhận xét kết quả.**

- Class distribution của CIFAR-10 chuẩn cân bằng giữa 10 classes.
- Sample images cho thấy CIFAR-10 khó hơn MNIST: ảnh nhỏ, nhiều màu và object/background đa dạng.
- Một sample có 3 RGB channels, nên Conv2D filter học trên cả channel màu lẫn không gian ảnh.

# 5. Data Preparation


Tạo train/validation/test split, fit channel mean/std chỉ trên train, normalize dữ liệu và tạo tensor layout riêng cho Keras/PyTorch.

**Nhận xét kết quả.**

- Original train split duoc tach thanh 45,000 train va 5,000 validation truoc khi lay subset runtime.
- Framework training dung 10,000 train va 3,000 validation de notebook chay duoc tren CPU/laptop.
- Channel mean/std duoc fit tren train split, sau do ap dung cho validation/test; test set van giu doc lap.

# 6. Discover CNN Operations


Minh họa Conv2D, ReLU, MaxPool, Flatten, Dense logits và softmax bằng tensor RGB nhỏ.

**Nhận xét kết quả.**

- Input có 3 channels RGB; mỗi Conv2D filter học weights trên cả 3 channels và tạo một feature map.
- Pooling giảm spatial size, Flatten chuyển feature maps sang vector, Dense tạo 10 logits.

# 7. Inspect CNN Building Blocks


Kiểm tra Basic CNN block, residual block và SE/channel-attention block bằng dummy tensor trước khi train model thật.

**Nhận xét kết quả.**

- Residual block đổi channels 3→16 và giảm spatial size bằng stride=2, nên shortcut dùng projection để shape khớp.
- SE block giữ nguyên tensor shape nhưng tạo channel weights để scale từng feature channel.

# 8. CNN From Scratch with NumPy


Định nghĩa NumPy CNN layers có forward/backward thật cho RGB CIFAR-10: Conv2D, ReLU, MaxPool, Flatten, Dense, SoftmaxCrossEntropy và SGD update.

**Nhận xét kết quả.**

- NumPy components có forward/backward thật cho Conv2D, ReLU, MaxPool2D, Flatten, Dense và SoftmaxCrossEntropy.
- `SGDNumPy` là optimizer update thủ công; phần này không dùng keras layers, torch layers hay autograd.
- Conv2D nhận RGB input nên weight shape bao gồm cả 3 input channels.

# 9. NumPy CNN Training


Test từng layer NumPy, train một CNN nhỏ trên stratified CIFAR-10 subset và lưu best state.

**Nhận xét kết quả.**

- Layer tests kiểm tra output shape, gradient shape, finite values, parameter shape và softmax row sums trước khi train.
- Numerical gradient check trên Conv2D xác nhận backward hợp lý trong một case nhỏ.
- NumPy CNN train thật trên subset 4,000 ảnh; best state được chọn theo validation Macro-F1 và lưu vào artifact.

# 10. Tensor Shape and Parameter Inspection


Tính parameter từng layer của NumPy CNN và kiểm tra manual calculation cho Conv layer đầu.

**Nhận xét kết quả.**

- Conv1 dùng kernel 3x3, 3 RGB input channels và 6 filters nên có `(3x3x3x6)+6` parameters.
- Parameter count nhỏ giúp NumPy CNN chạy được trên CPU nhưng vẫn thể hiện đúng CNN building blocks.

# 11. CNN with Keras/TensorFlow


Xây Keras Basic CNN cho CIFAR-10 với augmentation chỉ áp dụng trong training.

**Nhận xét kết quả.**

- Keras Basic CNN nhận `(B, 32, 32, 3)` và trả `(B, 10)` probabilities.
- Augmentation sẽ được đặt trong `tf.data` training pipeline ở cell sau để validation/test không đổi shape và không bị augment.

# 12. Keras Training


Train Keras Basic CNN với checkpoint, EarlyStopping và lưu training curves.

**Nhận xét kết quả.**

- Training dùng augmentation trên train only và validation data không augment.
- Curves giúp xem CIFAR-10 có underfit/overfit trong budget epoch ngắn hay không.

# 13. Keras Evaluation


Evaluate Keras Basic CNN trên full CIFAR-10 test set và lưu confusion matrix.

**Nhận xét kết quả.**

- Keras Basic CNN đạt Accuracy 50.09% và Macro-F1 49.84% trên full CIFAR-10 test set.
- Confusion matrix cho thấy model đã học được tín hiệu ảnh RGB, nhưng CIFAR-10 vẫn khó hơn MNIST nên lỗi giữa các lớp gần nhau còn rõ.
- Metrics được báo cáo theo macro average để mỗi class có trọng số ngang nhau.

# 14. CNN with PyTorch


Xây PyTorch Basic CNN tương đương ý tưởng với Keras và kiểm tra batch/logits shape trước training.

**Nhận xét kết quả.**

- PyTorch nhận CIFAR-10 theo layout `(B, 3, 32, 32)`.
- Logits shape `(B, 10)` cho thấy mỗi ảnh có 10 class scores trước softmax.

# 15. PyTorch Training


Train PyTorch Basic CNN bằng training loop rõ ràng và dùng augmentation chỉ trong train loop.

**Nhận xét kết quả.**

- Training loop có đủ `zero_grad`, forward, loss, backward và optimizer step.
- Augmentation chỉ nằm trong train loop; validation/test đi qua ảnh normalized không augment.

# 16. PyTorch Evaluation


Load PyTorch Basic CNN checkpoint tốt nhất và evaluate trên full test set.

**Nhận xét kết quả.**

- PyTorch Basic CNN dat Accuracy 49.62% va Macro-F1 49.55% voi architecture tuong duong y tuong voi Keras baseline.
- Loop train hien ro cac buoc `zero_grad`, forward logits, loss, backward va optimizer step.
- Khac biet so voi Keras trong mot lan chay khong du de ket luan framework nao tot hon; backend, augmentation va CPU runtime deu anh huong.

# 17. Framework Comparison


Evaluate NumPy CNN trên full test set và so sánh NumPy, Keras Basic CNN, PyTorch Basic CNN.

**Nhận xét kết quả.**

- Framework comparison tách riêng ba cách cài đặt cùng ý tưởng CNN cơ bản: NumPy, Keras và PyTorch.
- Keras Basic CNN có Macro-F1 49.84%, PyTorch Basic CNN có Macro-F1 49.55%, NumPy CNN có Macro-F1 25.65%.
- NumPy dùng 4,000 mẫu train, trong khi Keras/PyTorch dùng 10,000 mẫu; runtime phụ thuộc backend/hardware.

# 18. CNN Architecture Improvements


Dùng PyTorch làm framework chính để so M1 Basic CNN, M2 BatchNorm CNN, M3 Residual CNN và M4 Residual CNN + SE attention.

**Nhận xét kết quả.**

- M1-M4 dùng cùng split, normalization, augmentation, batch size, optimizer family và test set để báo cáo metrics.
- Bảng `selection_results` dùng validation Macro-F1 để chọn model, tách khỏi bảng test metrics.
- M3/M4 có residual connections; M4 thêm SE channel attention nhưng không được mặc định là tốt nhất trước khi xem output.

# 19. Improved Model Comparison


Kết hợp framework comparison và architecture evolution để chọn final candidate.

**Nhận xét kết quả.**

- Trong nhóm M1-M4, model tốt nhất là `M4 Residual CNN + SE Attention` với Macro-F1 47.39%.
- Bảng all-model cho thấy model đứng đầu toàn bộ run hiện tại là `Keras Basic CNN` với Macro-F1 49.84%.
- Kết quả không mặc định attention luôn tốt hơn; với budget ngắn, M4 tốt nhất trong nhóm architecture nhưng chưa vượt Keras Basic CNN.

# 20. Feature Map Inspection


Hiển thị original CIFAR images và feature maps từ early/deeper block của final PyTorch architecture candidate.

**Nhận xét kết quả.**

- Feature maps duoc lay tu M4 Residual CNN + SE Attention va luu thanh hinh `feature_maps.png`.
- Cac block som giu nhieu chi tiet cuc bo hon, trong khi block sau co spatial size nho hon va channel representation giau hon.
- Notebook khong gan nhan tuyet doi tung channel la mot detector cu the, vi feature map can duoc doc nhu tin hieu trung gian cua model.

# 21. Error Analysis


Tạo normalized confusion matrix, per-class accuracy và high-confidence mistakes của final candidate.

**Nhận xét kết quả.**

- Error analysis dùng final model đã được chọn bằng validation Macro-F1, rồi phân tích lỗi trên test set.
- Confusion matrix chuẩn hóa và per-class accuracy cho biết class nào yếu trong run hiện tại.
- High-confidence mistakes giúp xem những trường hợp model sai nhưng vẫn tự tin.

# 22. Final Model


Chọn final model theo Macro-F1, lưu model cuối và final metrics.

**Nhận xét kết quả.**

- Final model được chọn bằng validation Macro-F1, sau đó mới đọc test metrics tương ứng để báo cáo.
- Test set không được dùng để tune/chọn model; nó chỉ đóng vai trò đánh giá cuối.
- Model final đã được lưu trong `pipeline04/cifar10/models/` để notebook/report sau có thể tải lại.

# 23. Final Results


Lưu benchmark, metadata và liệt kê artifacts CIFAR-10 CNN.

**Nhận xét kết quả.**

- Bảng tổng hợp xếp 7 model theo Macro-F1; model đứng đầu là `Keras Basic CNN` với Macro-F1 49.84%.
- Tổng train time được ghi nhận trong artifacts là 256.3s; clean run-all CIFAR-10 cuối cùng mất 330.80s trong môi trường hiện tại.
- Artifacts đã gồm 8 model files, 26 metrics files và 13 figures.

In [ ]:
import pandas as pd
from IPython.display import display, Image
_folder = _repo / 'A04/artifacts/pipeline04/cifar10/metrics'
for _file in sorted(_folder.glob('*comparison*.csv')):
    print(_file.name)
    display(pd.read_csv(_file))


In [ ]:
display(Image(filename=str(_repo / 'A04/artifacts/pipeline04/cifar10/figures/keras_basic_training_curves.png')))


In [ ]:
display(Image(filename=str(_repo / 'A04/artifacts/pipeline04/cifar10/figures/pytorch_basic_training_curves.png')))
